# 11 — Governed Unity Catalog Function for Portfolio Concentration Risk

In Notebook 10, our Multi-Agent Orchestrator learned how to route work between specialized agents.

Now we add another enterprise pattern: **deterministic business logic implemented as a governed Unity Catalog function**.

### Business rule

> A security whose portfolio weight is greater than 10% is considered a concentration breach.

The key principle is:

> **The LLM decides when a policy check is needed. The governed function decides the policy result.**

```text
User Question → Claude / Orchestrator → UC Function → Portfolio Data → BREACH / OK → Claude explains
```

## 1. Install the required libraries

We use the Unity Catalog Anthropic integration so the registered UC function can be exposed directly as a Claude tool.

In [0]:
%pip install -q --upgrade "unitycatalog-anthropic[databricks]" anthropic
dbutils.library.restartPython()

## 2. Configure the notebook and select a portfolio

The function reads the canonical portfolio source from Notebook 08.

We select one real portfolio from the POC snapshot so the demo does not depend on a hardcoded portfolio ID.

In [0]:
CATALOG = "investment_ai"
PORTFOLIO_SOURCE = f"{CATALOG}.portfolio.portfolio_semantic_source"
FUNCTION_NAME = f"{CATALOG}.agent.check_portfolio_concentration"

MODEL_PROVIDER_SERVICE = "investment_ai.gateway.anthropic_claude"
MODEL = "claude-sonnet-5-5"

POC_AS_OF_DATE = "2026-06-30"
DEFAULT_THRESHOLD = 0.10

snapshot = spark.sql(
    f'''
    SELECT
        portfolio_id,
        MAX(portfolio_name) AS portfolio_name,
        SUM(market_value) AS portfolio_market_value
    FROM {PORTFOLIO_SOURCE}
    WHERE as_of_date = DATE '{POC_AS_OF_DATE}'
    GROUP BY portfolio_id
    ORDER BY portfolio_market_value DESC
    LIMIT 1
    '''
).first()

if snapshot is None:
    raise ValueError(
        f"No portfolio data found in {PORTFOLIO_SOURCE} for {POC_AS_OF_DATE}."
    )

SAMPLE_PORTFOLIO_ID = snapshot["portfolio_id"]

print("Portfolio source :", PORTFOLIO_SOURCE)
print("UC function      :", FUNCTION_NAME)
print("Sample portfolio :", SAMPLE_PORTFOLIO_ID)
print("Portfolio name   :", snapshot["portfolio_name"])
print("Threshold        :", DEFAULT_THRESHOLD)

## 3. Create the governed concentration-risk function

This is the core of the notebook.

The function accepts a portfolio ID, an as-of date, and a threshold. It aggregates positions at the security level, calculates each security's share of total portfolio market value, and returns `BREACH` or `OK`.

The policy calculation now lives in Unity Catalog rather than inside an LLM prompt.

In [0]:
create_function_sql = f"""
CREATE OR REPLACE FUNCTION {FUNCTION_NAME}(
    p_portfolio_id STRING COMMENT 'Portfolio identifier to evaluate.',
    p_as_of_date DATE COMMENT 'Portfolio snapshot date.',
    p_threshold DOUBLE DEFAULT 0.10 COMMENT 'Maximum allowed security weight as a decimal.'
)
RETURNS TABLE (
    portfolio_id STRING,
    portfolio_name STRING,
    as_of_date DATE,
    security_id STRING,
    company_name STRING,
    sector STRING,
    total_quantity DOUBLE,
    market_value DOUBLE,
    position_weight DOUBLE,
    concentration_threshold DOUBLE,
    risk_status STRING,
    excess_weight DOUBLE
)
LANGUAGE SQL
READS SQL DATA
COMMENT 'Evaluate security-level portfolio concentration against a configurable threshold.'
RETURN
WITH security_positions AS (
    SELECT
        portfolio_id,
        MAX(portfolio_name) AS portfolio_name,
        as_of_date,
        security_id,
        MAX(company_name) AS company_name,
        MAX(sector) AS sector,
        SUM(quantity) AS total_quantity,
        SUM(market_value) AS market_value
    FROM {PORTFOLIO_SOURCE}
    WHERE portfolio_id = p_portfolio_id
      AND as_of_date = p_as_of_date
    GROUP BY portfolio_id, as_of_date, security_id
),
weighted_positions AS (
    SELECT
        *,
        market_value / NULLIF(SUM(market_value) OVER (), 0) AS position_weight
    FROM security_positions
)
SELECT
    portfolio_id,
    portfolio_name,
    as_of_date,
    security_id,
    company_name,
    sector,
    total_quantity,
    market_value,
    position_weight,
    p_threshold AS concentration_threshold,
    CASE
        WHEN position_weight > p_threshold THEN 'BREACH'
        ELSE 'OK'
    END AS risk_status,
    GREATEST(position_weight - p_threshold, 0.0) AS excess_weight
FROM weighted_positions
ORDER BY position_weight DESC
"""

spark.sql(create_function_sql)
print(f"✅ Created governed function: {FUNCTION_NAME}")

## 4. Test the business rule directly

Before introducing an LLM, we verify the policy logic independently.

This proves that `BREACH` / `OK` comes from the governed function, not from model reasoning.

In [0]:
display(
    spark.sql(
        f'''
        SELECT
            security_id,
            company_name,
            position_weight,
            concentration_threshold,
            risk_status,
            excess_weight
        FROM {FUNCTION_NAME}(
            '{SAMPLE_PORTFOLIO_ID}',
            DATE '{POC_AS_OF_DATE}',
            {DEFAULT_THRESHOLD}
        )
        ORDER BY position_weight DESC
        '''
    )
)

## 5. Expose the UC function as a Claude tool

`DatabricksFunctionClient` executes the registered function, while `UCFunctionToolkit` converts it into an Anthropic-compatible tool.

The Unity Catalog function remains the source of truth for the parameters and business logic.

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from unitycatalog.ai.anthropic.toolkit import UCFunctionToolkit
from anthropic import Anthropic
from databricks.sdk import WorkspaceClient

function_client = DatabricksFunctionClient()

toolkit = UCFunctionToolkit(
    function_names=[FUNCTION_NAME],
    client=function_client,
)

workspace = WorkspaceClient()
auth_header = workspace.config.authenticate()["Authorization"]
databricks_token = auth_header.replace("Bearer ", "", 1)

UNITY_GATEWAY_ANTHROPIC_URL = (
    f"{workspace.config.host.rstrip('/')}/ai-gateway/anthropic"
)

anthropic_client = Anthropic(
    auth_token=databricks_token,
    base_url=UNITY_GATEWAY_ANTHROPIC_URL,
    default_headers={
        "Databricks-Model-Provider-Service": MODEL_PROVIDER_SERVICE,
    },
)

print("✅ UC function exposed as a Claude tool.")

## 6. Define the agent behavior and tool-use loop

Claude can decide whether a concentration check is needed, but it is instructed not to calculate the policy itself.

When the rule is required, Claude calls the governed UC function and explains the returned result.

In [0]:
from unitycatalog.ai.anthropic.utils import generate_tool_call_messages

RISK_INSTRUCTIONS = '''
You are the Investment AI Portfolio Risk Assistant.

For portfolio concentration questions, use the governed
check_portfolio_concentration Unity Catalog tool.

Rules:
- Do not calculate concentration yourself when the tool is available.
- Treat the tool output as the authoritative policy result.
- Clearly identify BREACH positions.
- Express weights and thresholds as percentages.
- Mention the portfolio snapshot date.
- If no threshold is provided, use 0.10.
- Do not invent portfolio data or policy outcomes.
'''

def run_concentration_risk_assistant(question: str, max_iterations: int = 5) -> str:
    messages = [{"role": "user", "content": question}]

    for _ in range(max_iterations):
        response = anthropic_client.messages.create(
            model=MODEL,
            max_tokens=2048,
            system=RISK_INSTRUCTIONS,
            messages=messages,
            tools=toolkit.tools,
        )

        if response.stop_reason != "tool_use":
            return "\n".join(
                block.text
                for block in response.content
                if block.type == "text"
            )

        messages = generate_tool_call_messages(
            response=response,
            client=function_client,
            conversation_history=messages,
        )

    return "Reached the maximum number of tool-call iterations."

print("✅ Risk assistant ready.")

## 7. Test the complete AI + governed-function flow

Now we ask the question in normal business language.

Claude should recognize that a policy check is required, invoke the UC function, and explain the governed result.

In [0]:
question = (
    f"Check portfolio {SAMPLE_PORTFOLIO_ID} for concentration risk "
    f"as of {POC_AS_OF_DATE}. Use a 10% threshold. "
    "Which holdings breach the policy and by how much?"
)

answer = run_concentration_risk_assistant(question)
print(answer)

## 8. Where this fits in our architecture

We now have three different capability patterns:

```text
Unstructured research
Research Agent → AI Search

Flexible structured analytics
Portfolio Agent → Genie + Metric View

Deterministic business policy
Risk Tool → Unity Catalog Function
```

Genie is useful for flexible analytical questions.

The UC function is useful when the business already has an exact rule that must be executed consistently.

In production, access to the function can be governed through Unity Catalog permissions.

### Key takeaway

> **Use the LLM to decide what capability is needed. Use governed code to enforce deterministic business rules.**

“So with Notebook 11 complete, we now have our governed deterministic risk layer in place. In the next notebook, we’ll add an intelligent decision layer using ai_decide, where we can combine signals from research, portfolio exposure, and risk policy to decide whether something needs escalation or further review.”

Next: — MLflow Agent Evaluation and Quality.**